# Interview Prep Coach

A multi-agent interview preparation system built with LangGraph.

A worker agent researches a company/role using web search and Wikipedia,
generates interview questions, then an evaluator agent scores the user's
practice answers and loops back if they need improvement.

In [ ]:
from typing import Annotated, List, Any, Optional, Dict
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain.agents import Tool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
import uuid

load_dotenv(override=True)

In [ ]:
class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Feedback on the user's interview answer")
    score: int = Field(description="Score from 1-10 on answer quality")
    needs_improvement: bool = Field(description="Whether the answer needs significant improvement")
    user_input_needed: bool = Field(description="True if the coach needs more input or clarification from the user")


class State(TypedDict):
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool

In [ ]:
serper = GoogleSerperAPIWrapper()
tool_search = Tool(
    name="search",
    func=serper.run,
    description="Search the web for information about a company, role, or interview topics"
)

wikipedia = WikipediaAPIWrapper()
tool_wiki = WikipediaQueryRun(api_wrapper=wikipedia)

tools = [tool_search, tool_wiki]

In [ ]:
worker_llm = ChatOpenAI(model="gpt-4o-mini")
worker_llm_with_tools = worker_llm.bind_tools(tools)

evaluator_llm = ChatOpenAI(model="gpt-4o-mini")
evaluator_llm_with_output = evaluator_llm.with_structured_output(EvaluatorOutput)


def worker(state: State) -> Dict[str, Any]:
    system_message = f"""You are an interview prep coach with access to web search and Wikipedia tools.

When a user tells you a company and role, first research the company using your tools,
then generate 5 tailored interview questions based on your research.

When the user answers a question, provide detailed feedback and suggest improvements.

Success criteria: {state['success_criteria']}

If you have a question for the user, clearly state it.
Otherwise reply with your final answer."""

    if state.get("feedback_on_work"):
        system_message += f"""
Previously your response was rejected. Feedback: {state['feedback_on_work']}
Please improve your response based on this feedback."""

    messages = state["messages"]
    found_system = False
    for msg in messages:
        if isinstance(msg, SystemMessage):
            msg.content = system_message
            found_system = True
    if not found_system:
        messages = [SystemMessage(content=system_message)] + messages

    response = worker_llm_with_tools.invoke(messages)
    return {"messages": [response]}


def worker_router(state: State) -> str:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "evaluator"


def format_conversation(messages: List[Any]) -> str:
    conversation = ""
    for msg in messages:
        if isinstance(msg, HumanMessage):
            conversation += f"User: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            text = msg.content or "[Tool use]"
            conversation += f"Coach: {text}\n"
    return conversation


def evaluator(state: State) -> State:
    last_response = state["messages"][-1].content

    system_message = """You are an evaluator for an interview prep coach.
Assess whether the coach's response is helpful, accurate, and meets the success criteria.
Score the response and decide if it needs improvement."""

    user_message = f"""Conversation so far:
{format_conversation(state['messages'])}

Success criteria: {state['success_criteria']}

Coach's latest response: {last_response}

Evaluate this response. Decide if success criteria are met or if more user input is needed."""

    if state.get("feedback_on_work"):
        user_message += f"\nPrior feedback: {state['feedback_on_work']}"
        user_message += "\nIf the coach keeps repeating mistakes, set user_input_needed to true."

    eval_result = evaluator_llm_with_output.invoke([
        SystemMessage(content=system_message),
        HumanMessage(content=user_message),
    ])

    return {
        "messages": [{"role": "assistant", "content": f"Evaluator (Score: {eval_result.score}/10): {eval_result.feedback}"}],
        "feedback_on_work": eval_result.feedback,
        "success_criteria_met": not eval_result.needs_improvement,
        "user_input_needed": eval_result.user_input_needed,
    }


def route_after_eval(state: State) -> str:
    if state["success_criteria_met"] or state["user_input_needed"]:
        return "END"
    return "worker"

In [ ]:
graph_builder = StateGraph(State)

graph_builder.add_node("worker", worker)
graph_builder.add_node("tools", ToolNode(tools=tools))
graph_builder.add_node("evaluator", evaluator)

graph_builder.add_conditional_edges("worker", worker_router, {"tools": "tools", "evaluator": "evaluator"})
graph_builder.add_edge("tools", "worker")
graph_builder.add_conditional_edges("evaluator", route_after_eval, {"worker": "worker", "END": END})
graph_builder.add_edge(START, "worker")

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
thread_id = str(uuid.uuid4())


async def process_message(message, success_criteria, history, thread):
    config = {"configurable": {"thread_id": thread}}
    state = {
        "messages": message,
        "success_criteria": success_criteria or "Provide well-researched, tailored interview questions and helpful feedback",
        "feedback_on_work": None,
        "success_criteria_met": False,
        "user_input_needed": False,
    }
    result = await graph.ainvoke(state, config=config)
    user = {"role": "user", "content": message}
    reply = {"role": "assistant", "content": result["messages"][-2].content}
    feedback = {"role": "assistant", "content": result["messages"][-1].content}
    return history + [user, reply, feedback]


async def reset():
    return "", "", None, str(uuid.uuid4())


with gr.Blocks(theme=gr.themes.Default(primary_hue="blue")) as demo:
    gr.Markdown("## Interview Prep Coach")
    thread = gr.State(thread_id)

    chatbot = gr.Chatbot(label="Coach", height=400, type="messages")
    with gr.Group():
        message = gr.Textbox(show_label=False, placeholder="e.g. I'm interviewing for a backend engineer role at Stripe")
        success_criteria = gr.Textbox(show_label=False, placeholder="Success criteria (optional)")
    with gr.Row():
        reset_button = gr.Button("Reset", variant="stop")
        go_button = gr.Button("Go!", variant="primary")

    message.submit(process_message, [message, success_criteria, chatbot, thread], [chatbot])
    go_button.click(process_message, [message, success_criteria, chatbot, thread], [chatbot])
    reset_button.click(reset, [], [message, success_criteria, chatbot, thread])

demo.launch()